In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from master_thesis.deep_learning.explainability.global_analysis.representations import (
    extract_rnn_representations,
    compute_pca_representation,
)
from master_thesis.deep_learning.explainability.pretraining.local import explain_patient
import os
import re
from master_thesis.deep_learning.utils import load_config
from master_thesis.deep_learning.data import ECGDataset
from master_thesis.deep_learning.models.ecg_model import ECGModel
from torch.utils.data import DataLoader
import torch
from master_thesis.deep_learning.explainability.visualization import plot_local_explanation
from matplotlib import pyplot as plt
import shap
import torch.nn as nn
import pandas as pd
from typing import Any
from tqdm import tqdm
import gc

from master_thesis.deep_learning.explainability.pretraining.utils import (
    find_prediction_cases,
    get_representative_cases,
)
from master_thesis.deep_learning.explainability.global_analysis.leads import (
    compute_global_lead_importance,
)
from master_thesis.deep_learning.explainability.visualization import plot_global_lead_importance
from master_thesis.deep_learning.explainability.common.shap import (
    compute_tabular_shap
)
from master_thesis.constants import COLUMNS
from master_thesis.deep_learning.explainability.common.shap import (
    compute_global_tabular_shap
)
from master_thesis.deep_learning.explainability.common.grad_cam import (
    compute_cnn_grad_cam,
)
from master_thesis.deep_learning.explainability.visualization import (
    plot_cnn_grad_cam,
)

from master_thesis.deep_learning.explainability.visualization import (
    plot_rnn_representation,
)
import numpy as np

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# loading all models for 5 seeds both for pretrained and finetuned models
checkpoint_dirs = os.listdir("../checkpoints")
bigru_pattern = re.compile(r"^bigru_fusion_(?:pretrain|finetune)_\d+$")
cnn_lstm_pattern = re.compile(r"^cnn_lstm_fusion_(?:pretrain|finetune)_\d+$")
matching_dirs = [
    d for d in checkpoint_dirs
    if bigru_pattern.match(d) or cnn_lstm_pattern.match(d)
]
models = {
    "model_type": [],
    "train_stage": [],
    "seed": [],
    "model_path": [],
}
for dir_name in matching_dirs:
    model_type = "gru" if "bigru" in dir_name else "cnn_lstm"
    train_stage = "pretrain" if "pretrain" in dir_name else "finetune"
    seed = int(dir_name.split("_")[-1])
    model_path = f"../checkpoints/{dir_name}/best.pt"
    models["model_type"].append(model_type)
    models["train_stage"].append(train_stage)
    models["seed"].append(seed)
    models["model_path"].append(model_path)

In [5]:
SELECTED_SEED = 123

In [6]:
bigru_pretrain_config = load_config("../configs/15_bigru_fusion_pretrain_123.yaml")
bigru_finetune_config = load_config("../configs/20_bigru_fusion_finetune_123.yaml")
cnn_lstm_pretrain_config = load_config(
    "../configs/25_cnn_lstm_fusion_pretrain_123.yaml"
)
cnn_lstm_finetune_config = load_config(
    "../configs/30_cnn_lstm_fusion_finetune_123.yaml"
)

CONFIG_MAP = {
    ("gru", "pretrain"): bigru_pretrain_config,
    ("gru", "finetune"): bigru_finetune_config,
    ("cnn_lstm", "pretrain"): cnn_lstm_pretrain_config,
    ("cnn_lstm", "finetune"): cnn_lstm_finetune_config,
}

common_tabular_features = [
    "sex",
    "ventricular_rate",
    "atrial_rate",
    "pr_interval",
    "qrs_duration",
    "qt_corrected",
    "age_at_ecg",
]

In [7]:
dataset_args = {
    "waveforms_path": "../data/EchoNext_test_waveforms.npy",
    "metadata_path": "../data/echonext_metadata_100k.csv",
    "split": "test",
    "tabular_path": "../data/EchoNext_test_tabular_features.npy",
    "tabular_features": common_tabular_features,
}
val_args = {
    "waveforms_path": "../data/EchoNext_val_waveforms.npy",
    "metadata_path": "../data/echonext_metadata_100k.csv",
    "split": "val",
    "tabular_path": "../data/EchoNext_val_tabular_features.npy",
    "tabular_features": common_tabular_features,
}
batch_size = 32
num_workers = 4

pretrain_dataset = ECGDataset(**dataset_args, pretrain=True)
pretrain_loader = DataLoader(
    pretrain_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
)

finetune_dataset = ECGDataset(**dataset_args, pretrain=False)
finetune_loader = DataLoader(
    finetune_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
)

In [8]:
pretrain_diseases = list(pretrain_dataset.label_columns)
finetune_diseases = [
    d
    for d in finetune_dataset.label_columns
    if d != "shd_moderate_or_greater_flag"
]

In [9]:
# calibrate finetuned models on validation set
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

def fit_multilabel_isotonic_scalers(val_logits, val_y_true):
    val_logits = np.asarray(val_logits)
    val_y_true = np.asarray(val_y_true)
    n_diseases = val_logits.shape[1]
    
    scalers = []
    for d in range(n_diseases):
        y_col = val_y_true[:, d]
        logit_col = val_logits[:, d]
        
        if len(np.unique(y_col)) > 1:
            iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
            iso.fit(logit_col, y_col)
            scalers.append(iso)
        else:
            scalers.append(None)
            
    return scalers

def transform_with_isotonic(logits, scalers):
    logits = np.asarray(logits)
    calibrated_probs = np.zeros_like(logits, dtype=float)
    
    for d, iso in enumerate(scalers):
        if iso is not None:
            calibrated_probs[:, d] = iso.predict(logits[:, d])
        else:
            # fallback to sigmoid if class was degenerate
            calibrated_probs[:, d] = 1.0 / (1.0 + np.exp(-np.clip(logits[:, d], -50, 50)))
            
    return calibrated_probs

In [12]:
def plot_all_local_explanations(
    model_idx,
    model_type,
    train_stage,
    disease,
    patient_id,
    save_dir="../explanations",
):
    os.makedirs(save_dir, exist_ok=True)
    model_path = models["model_path"][model_idx]
    config = CONFIG_MAP[(model_type, train_stage)]
    loader = (
        pretrain_loader
        if train_stage == "pretrain"
        else finetune_loader
    )
    diseases = (
        pretrain_diseases
        if train_stage == "pretrain"
        else finetune_diseases
    )
    model = ECGModel(
        diseases=diseases,
        config=config
    )
    checkpoint = torch.load(
        model_path,
        map_location=device
    )
    model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    model.to(device)
    model.eval()
    threshold = thresholds.get(
        (model_type, disease),
        0.5
    )
    explanation = explain_patient(
        model=model,
        dataset=loader.dataset,
        index=patient_id,
        target=disease,
        device=device,
        threshold=threshold
    )
    fig = plot_local_explanation(
        result=explanation,
        waveform=loader.dataset[patient_id]["waveform"],
    )
    fig.savefig(
        os.path.join(
            save_dir,
            f"{model_type}_{train_stage}_{disease}_patient_{patient_id}.png"
        ),
        bbox_inches="tight"
    )

In [10]:
thresholds = {
    ("gru", "shd_moderate_or_greater_flag"): 0.4,
    ("gru", "aortic_regurgitation_moderate_or_greater_flag"): 0.05,
    ("gru", "aortic_stenosis_moderate_or_greater_flag"): 0.05,
    ("gru", "lvef_lte_45_flag"): 0.25,
    ("gru", "lvwt_gte_13_flag"): 0.2,
    ("gru", "mitral_regurgitation_moderate_or_greater_flag"): 0.05,
    ("gru", "pasp_gte_45_flag"): 0.1,
    ("gru", "pericardial_effusion_moderate_large_flag"): 0.05,
    ("gru", "pulmonary_regurgitation_moderate_or_greater_flag"): 0.05,
    ("gru", "rv_systolic_dysfunction_moderate_or_greater_flag"): 0.1,
    ("gru", "tr_max_gte_32_flag"): 0.05,
    ("gru", "tricuspid_regurgitation_moderate_or_greater_flag"): 0.05,

    ("cnn_lstm", "shd_moderate_or_greater_flag"): 0.4,
    ("cnn_lstm", "aortic_regurgitation_moderate_or_greater_flag"):  0.05,
    ("cnn_lstm", "aortic_stenosis_moderate_or_greater_flag"): 0.05,
    ("cnn_lstm", "lvef_lte_45_flag"): 0.3,
    ("cnn_lstm", "lvwt_gte_13_flag"): 0.2,
    ("cnn_lstm", "mitral_regurgitation_moderate_or_greater_flag"): 0.05,
    ("cnn_lstm", "pasp_gte_45_flag"): 0.1,
    ("cnn_lstm", "pericardial_effusion_moderate_large_flag"): 0.05,
    ("cnn_lstm", "pulmonary_regurgitation_moderate_or_greater_flag"): 0.05,
    ("cnn_lstm", "rv_systolic_dysfunction_moderate_or_greater_flag"): 0.15,
    ("cnn_lstm", "tr_max_gte_32_flag"): 0.05,
    ("cnn_lstm", "tricuspid_regurgitation_moderate_or_greater_flag"): 0.1,
}

In [11]:
def predict_single_model(
    model: nn.Module,
    loader: DataLoader,
    diseases: list[str],
    use_tabular: bool = False,
    return_logits: bool = False,
    device: torch.device | None = None,
) -> np.ndarray:
    """runs inference for a single model instance over a dataLoader"""
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    batch_outputs = []

    with torch.no_grad():
        for batch in loader:
            waveforms = batch["waveform"].to(device, non_blocking=True)
            tabular_data = batch.get("tabular")

            if tabular_data is not None and use_tabular:
                tabular_data = tabular_data.to(device, non_blocking=True)
            else:
                tabular_data = None

            out = model(waveforms, tabular_data)

            if isinstance(out, tuple):
                outputs_dict = out[0]
            elif isinstance(out, dict):
                outputs_dict = out
            else:
                batch_outputs.append(
                    out.cpu().numpy()
                    if return_logits
                    else torch.sigmoid(out).cpu().numpy()
                )
                continue

            logits = torch.stack(
                [outputs_dict[d].squeeze(-1) for d in diseases], dim=-1
            )
            vals = (
                logits if return_logits else torch.sigmoid(logits)
            ).detach().cpu().numpy()
            batch_outputs.append(vals)

    return np.concatenate(batch_outputs, axis=0)


def run_inference(
    dataset_args: dict,
    models: dict | None = None,
    instantiated_models: list[nn.Module] | None = None,
    config_map: dict | None = None,
    batch_size: int = 32,
    num_workers: int = 0,
    return_logits: bool = False,
    device: torch.device | None = None,
) -> dict[str, list[Any]]:
    """runs batch inference over a dataset split

    supports either:
      1. passing instantiated_models directly
      2. passing `models` dict containing checkpoint paths and metadata to load
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    pretrain_dataset = ECGDataset(**dataset_args, pretrain=True)
    pretrain_loader = DataLoader(
        pretrain_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    finetune_dataset = ECGDataset(**dataset_args, pretrain=False)
    finetune_loader = DataLoader(
        finetune_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    pretrain_diseases = list(pretrain_dataset.label_columns)
    finetune_diseases = [
        d
        for d in finetune_dataset.label_columns
        if d != "shd_moderate_or_greater_flag"
    ]

    split_name = dataset_args.get("split", "dataset")
    results = {
        "model_type": [],
        "train_stage": [],
        "seed": [],
        "predictions": [],
    }

    total_runs = len(models["model_path"]) if models is not None else len(instantiated_models)

    for i in tqdm(range(total_runs), desc=f"Predicting on '{split_name}' set"):
        model_type = models["model_type"][i]
        train_stage = models["train_stage"][i]
        seed = models["seed"][i]

        loader = pretrain_loader if train_stage == "pretrain" else finetune_loader
        diseases = pretrain_diseases if train_stage == "pretrain" else finetune_diseases

        config = config_map[(model_type, train_stage)]
        use_tabular = config.get("tabular", {}).get("enabled", False)

        if instantiated_models is not None:
            model = instantiated_models[i].to(device)
            owns_model = False
        else:
            model = ECGModel(diseases=diseases, config=config)
            checkpoint = torch.load(models["model_path"][i], map_location=device)
            state_dict = (
                checkpoint["model_state_dict"]
                if "model_state_dict" in checkpoint
                else checkpoint
            )
            model.load_state_dict(state_dict)
            model.to(device)
            owns_model = True

        preds = predict_single_model(
            model=model,
            loader=loader,
            diseases=diseases,
            use_tabular=use_tabular,
            return_logits=return_logits,
            device=device,
        )

        results["model_type"].append(model_type)
        results["train_stage"].append(train_stage)
        results["seed"].append(seed)
        results["predictions"].append(preds)

        if owns_model:
            del model, checkpoint, state_dict
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return results

In [14]:
val_logits_dict = run_inference(
    dataset_args=val_args,
    models=models,
    config_map=CONFIG_MAP,
    return_logits=True,
)

Predicting on 'val' set: 100%|██████████| 20/20 [00:42<00:00,  2.11s/it]


In [16]:
val_finetune = ECGDataset(**val_args, pretrain=False)

In [17]:
test_logits_dict = run_inference(
    dataset_args=dataset_args,
    models=models,
    config_map=CONFIG_MAP,
    return_logits=True,
)

Predicting on 'test' set: 100%|██████████| 20/20 [00:48<00:00,  2.42s/it]


In [ ]:
for i in range(len(models["model_type"])):
    if models["seed"][i] != SELECTED_SEED:
        continue

    model_type = models["model_type"][i]
    train_stage = models["train_stage"][i]
    diseases = (
        pretrain_diseases
        if train_stage == "pretrain"
        else finetune_diseases
    )

    if train_stage == "finetune":
        val_logits = val_logits_dict["predictions"][i]
        val_true = val_finetune.labels
        scalers = fit_multilabel_isotonic_scalers(
            val_logits,
            val_true
        )
        test_logits = test_logits_dict["predictions"][i]
        calibrated_probs = transform_with_isotonic(
            test_logits,
            scalers
        )
    else:
        test_logits = test_logits_dict["predictions"][i]

    for disease in diseases:
        threshold = thresholds.get(
            (model_type, disease),
            0.5
        )

        if train_stage == "finetune":
            disease_idx = finetune_diseases.index(disease)
            y_true_disease = (
                finetune_dataset.labels[:, disease_idx]
            )
            y_probs = calibrated_probs[:, disease_idx]
        else:
            disease_idx = pretrain_diseases.index(disease)
            y_true_disease = (
                pretrain_dataset.labels[:, disease_idx]
            )
            y_probs = test_logits[:, disease_idx]

        prediction_cases = find_prediction_cases(
            y_true=y_true_disease,
            y_probs=y_probs,
            threshold=threshold
        )
        representative_cases = get_representative_cases(
            prediction_cases,
            n=1,
            threshold=threshold
        )

        for case_type in representative_cases.keys():
            for case in representative_cases[case_type]:
                patient_id = case["index"]
                plot_all_local_explanations(
                    i,
                    model_type,
                    train_stage,
                    disease,
                    patient_id,
                    save_dir="../explanations/local_explanations"
                )

In [ ]:
# global lead importance for all diseases and model types
os.makedirs("../explanations/global_lead_importance", exist_ok=True)
for i in range(len(models["model_type"])):
    if models["seed"][i] != SELECTED_SEED:
        continue

    model_type = models["model_type"][i]
    train_stage = models["train_stage"][i]
    diseases = (
        pretrain_diseases
        if train_stage == "pretrain"
        else finetune_diseases
    )

    model_path = models["model_path"][i]
    config = CONFIG_MAP[(model_type, train_stage)]
    loader = (
        pretrain_loader
        if train_stage == "pretrain"
        else finetune_loader
    )
    model = ECGModel(
        diseases=diseases,
        config=config
    )
    checkpoint = torch.load(
        model_path,
        map_location=device
    )
    model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    model.to(device)
    model.eval()

    for disease in tqdm(diseases, desc=f"Global lead importance for {model_type} {train_stage}"):
        threshold = thresholds.get(
            (model_type, disease),
            0.5
        )
        global_lead_importance = compute_global_lead_importance(
            model=model,
            dataset=loader.dataset,
            target=disease,
            device=device
        )
        fig = plot_global_lead_importance(
            global_lead_importance,
            title=f"{model_type} {train_stage} - {disease} - Global Lead Importance"
        )
        fig.savefig(
            os.path.join(
                "../explanations/global_lead_importance",
                f"{model_type}_{train_stage}_{disease}_global_lead_importance.png"
            ),
            bbox_inches="tight"
        )

In [13]:
# global shap for tabular features for all diseases and model types
os.makedirs("../explanations/global_tabular_shap", exist_ok=True)
for i in range(len(models["model_type"])):
    if models["model_type"][i] != "cnn_lstm":
        continue
    if models["seed"][i] != SELECTED_SEED:
        continue

    model_type = models["model_type"][i]
    train_stage = models["train_stage"][i]
    diseases = (
        pretrain_diseases
        if train_stage == "pretrain"
        else finetune_diseases
    )

    model_path = models["model_path"][i]
    config = CONFIG_MAP[(model_type, train_stage)]
    loader = (
        pretrain_loader
        if train_stage == "pretrain"
        else finetune_loader
    )
    model = ECGModel(
        diseases=diseases,
        config=config
    )
    checkpoint = torch.load(
        model_path,
        map_location=device
    )
    model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    model.to(device)
    model.eval()

    for disease in tqdm(diseases, desc=f"Global tabular SHAP for {model_type} {train_stage}"):
        threshold = thresholds.get(
            (model_type, disease),
            0.5
        )
        global_tabular_shap = compute_global_tabular_shap(
            model=model,
            dataset=loader.dataset,
            target=disease,
            device=device,
            background_indices=np.random.choice(len(loader.dataset), size=100, replace=False),
            explain_indices=np.random.choice(len(loader.dataset), size=1000, replace=False)
        )
        explanation = shap.Explanation(
            values=global_tabular_shap["shap_values"],
            data=global_tabular_shap["tabular_values"],
            feature_names=COLUMNS,
        )
        
        shap.plots.beeswarm(
            explanation,
            max_display=len(COLUMNS),
            show=False,
        )
        plt.title(f"{model_type} {train_stage} - {disease} - Global Tabular SHAP")
        plt.savefig(
            os.path.join(
                "../explanations/global_tabular_shap",
                f"{model_type}_{train_stage}_{disease}_global_tabular_shap.png"
            ),
            bbox_inches="tight"
        )
        plt.close()

Global tabular SHAP for cnn_lstm pretrain: 100%|██████████| 1/1 [02:02<00:00, 122.75s/it]


In [ ]:
index = 0
sample = pretrain_dataset[index]
waveform = sample["waveform"].unsqueeze(0)
tabular = sample.get("tabular")

os.makedirs("../explanations/cnn_grad_cam", exist_ok=True)

if tabular is not None:
    tabular = tabular.unsqueeze(0)

for i in range(len(models["model_type"])):
    if models["seed"][i] != SELECTED_SEED:
        continue
    if models["model_type"][i] != "cnn_lstm":
        continue

    train_stage = models["train_stage"][i]
    diseases = (
        pretrain_diseases
        if train_stage == "pretrain"
        else finetune_diseases
    )

    model_type = models["model_type"][i]
    train_stage = models["train_stage"][i]

    model_path = models["model_path"][i]
    config = CONFIG_MAP[(model_type, train_stage)]
    model = ECGModel(
        diseases=diseases,
        config=config
    )
    checkpoint = torch.load(
        model_path,
        map_location=device
    )
    model.load_state_dict(torch.load(model_path, map_location=device)["model_state_dict"])
    model.to(device)
    model.eval()

    for disease in tqdm(diseases, desc=f"CNN Grad-CAM for {model_type} {train_stage}"):
        threshold = thresholds.get(
            (model_type, disease),
            0.5
        )
        grad_cam_result = compute_cnn_grad_cam(
            model=model,
            waveform=waveform.to(device),
            tabular=tabular.to(device) if tabular is not None else None,
            target=disease,
            device=device
        )
        fig = plot_cnn_grad_cam(
            grad_cam_result,
            waveform=waveform.squeeze(0).cpu().numpy()
        )
        fig.savefig(
            os.path.join(
                "../explanations/cnn_grad_cam",
                f"{model_type}_{train_stage}_{disease}_cnn_grad_cam.png"
            ),
            bbox_inches="tight"
        )

In [ ]:
os.makedirs("../explanations/rnn_representations", exist_ok=True)

for i in range(len(models["model_type"])):
    if models["seed"][i] != SELECTED_SEED:
        continue

    train_stage = models["train_stage"][i]
    diseases = (
        pretrain_diseases
        if train_stage == "pretrain"
        else finetune_diseases
    )

    model_type = models["model_type"][i]
    train_stage = models["train_stage"][i]

    model_path = models["model_path"][i]
    config = CONFIG_MAP[(model_type, train_stage)]
    model = ECGModel(
        diseases=diseases,
        config=config
    )
    checkpoint = torch.load(
        model_path,
        map_location=device
    )
    model.load_state_dict(torch.load(model_path, map_location=device)["model_state_dict"])
    model.to(device)
    model.eval()

    for disease in tqdm(diseases, desc=f"RNN Representations for {model_type} {train_stage}"):
        threshold = thresholds.get(
            (model_type, disease),
            0.5
        )
        representations = extract_rnn_representations(
            model=model,
            dataset=pretrain_dataset if train_stage == "pretrain" else finetune_dataset,
            target=disease,
            device=device
        )
        pca_result = compute_pca_representation(representations)
        fig = plot_rnn_representation(
            pca_result
        )
        fig.savefig(
            os.path.join(
                "../explanations/rnn_representations",
                f"{model_type}_{train_stage}_{disease}_rnn_representations.png"
            ),
            bbox_inches="tight"
        )

In [12]:
import joblib
from pathlib import Path

CALIBRATION_DIR = Path("../calibration")
CALIBRATION_DIR.mkdir(exist_ok=True)

In [17]:
for i in range(len(models["model_type"])):
    if models["seed"][i] != SELECTED_SEED:
        continue

    model_type = models["model_type"][i]
    train_stage = models["train_stage"][i]
    diseases = (
        pretrain_diseases
        if train_stage == "pretrain"
        else finetune_diseases
    )

    if train_stage == "finetune":
        val_logits = val_logits_dict["predictions"][i]
        val_true = val_finetune.labels
        scalers = fit_multilabel_isotonic_scalers(
            val_logits,
            val_true
        )
        calibration_path = (
            CALIBRATION_DIR
            / f"isotonic_{model_type}_seed_{SELECTED_SEED}.joblib"
        )

        joblib.dump(
            scalers,
            calibration_path,
        )